In [1]:
import ast
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
from tqdm import tqdm

for genus_name in keep_genus:
    basic_dir = rf'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
    replicon_data = pd.read_csv(f'{basic_dir}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    ori_results = replicon_data[['accession', 'size', 'average plasmid fraction-pident_90', 'ms-label', 'category-pident_90',]].copy()
    with tqdm(total = len(ori_results), desc=f'{genus_name}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for i, row in ori_results.iterrows():
            acc_n, contig = row['accession'].split('-')
            try:
                temp_oric = pd.read_csv(f'{basic_dir}/Ori_finder/oriC_ORCA/{acc_n}/{contig}.csv')
                if max(temp_oric['predictions']) >= 0.5:
                    ori_results.loc[i, 'oriC(ORCA)'] = 'Y'
                else:
                    ori_results.loc[i, 'oriC(ORCA)'] = 'N'
            except:
                ori_results.loc[i, 'oriC(ORCA)'] = 'N'
            if row['ms-label'] == 'NMS_replicon':
                try:
                    temp_oriv = pd.read_csv(f'{basic_dir}/Ori_finder/orivfinder/{acc_n}/{contig}/selected_ori_regions.csv')
                    if max(temp_oriv['Total_Score']) >= 200:
                        ori_results.loc[i, 'oriV(OriV-Finder)'] = 'Y'
                    else:
                        ori_results.loc[i, 'oriV(OriV-Finder)'] = 'N'
                except:
                    ori_results.loc[i, 'oriV(OriV-Finder)'] = 'N'
            else:
                ori_results.loc[i, 'oriV(OriV-Finder)'] = 'Skipped'
            pbar.update(1)
        ori_results.to_csv(f'{basic_dir}/Ori_finder/ori_results.csv', index=False)

Escherichia: 100%|███████████████████████████████████████████████| 15.4k/15.4k [00:36<00:00, 426B/s]
Klebsiella: 100%|████████████████████████████████████████████████| 15.0k/15.0k [00:35<00:00, 420B/s]
Staphylococcus: 100%|████████████████████████████████████████████| 5.08k/5.08k [00:10<00:00, 499B/s]
Pseudomonas: 100%|███████████████████████████████████████████████| 3.14k/3.14k [00:04<00:00, 659B/s]
Bacillus: 100%|██████████████████████████████████████████████████| 3.99k/3.99k [00:07<00:00, 530B/s]
Salmonella: 100%|████████████████████████████████████████████████| 4.32k/4.32k [00:08<00:00, 483B/s]
Streptococcus: 100%|█████████████████████████████████████████████| 1.78k/1.78k [00:02<00:00, 783B/s]
Streptomyces: 100%|██████████████████████████████████████████████| 2.46k/2.46k [00:04<00:00, 551B/s]
Acinetobacter: 100%|█████████████████████████████████████████████| 3.74k/3.74k [00:08<00:00, 450B/s]
Enterococcus: 100%|██████████████████████████████████████████████| 3.29k/3.29k [00:07<00:00